# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset defined by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
 - https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata object directly
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Schema conforms to: {metadata.conforms_to}")
print(f"Published on: {metadata.date_published}")
print(f"Authors (@id): {[a['@id'] if isinstance(a, dict) and '@id' in a else a for a in metadata.author]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the available record sets and print their `@id`s. Then for each record set, we list their fields and columns via `@id`.

In [ ]:
# List available record sets, fields, and columns

record_sets = dataset.record_sets()
print("Record sets available:")
for rs in record_sets:
    print(f"- Record Set name: {rs.name}")
    print(f"  @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}, @id: {field['@id']}, Data type: {field.data_type}")
        if hasattr(field, 'column') and field.column is not None:
            print(f"      Column @id: {field.column['@id']} - {field.column.name if hasattr(field.column, 'name') else field.column['@id']}")
    print("\n---")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis.

All references below use the record set, field, and column `@id`s as required.

In [ ]:
# Extract data from each record set by @id
# Get all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
print(f"Record Set IDs: {record_set_ids}")
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for record set @id `{record_set_id}`:")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# Select the first record set for demonstration (adjust as needed)
main_record_set_id = record_set_ids[0]
df_main = dataframes[main_record_set_id]
print(f"Sample records from `{main_record_set_id}`:")
df_main.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering, normalizing, grouping. Use field and column `@id`s throughout.

In [ ]:
# Identify a numeric field by @id from previous overview
# Common fields: 'gad_7_score', 'phq_9_score', etc. Assume their @id is known from previous cells.
# For demonstration, let's filter GAD-7 scores above threshold and normalize them

numeric_field_id = 'gad_7_score' # Use the actual @id if available, otherwise use the column name

# Check field presence
if numeric_field_id in df_main.columns:
    threshold = 10
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with `{numeric_field_id}` > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized `{numeric_field_id}` for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a demographic field, such as 'level_of_education' (assume its @id is used)
    group_field_id = 'level_of_education'
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean `{numeric_field_id}` by `{group_field_id}`:")
        print(grouped_df)
else:
    print(f"Field `{numeric_field_id}` not found in main record set. Available columns: {df_main.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Examples:
 - Distribution of GAD-7 scores
 - Boxplot by education level


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric_field_id (GAD-7 score)
if numeric_field_id in df_main.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (GAD-7 Score)")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field_id
    if group_field_id in df_main.columns:
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df_main, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel("Level of Education")
        plt.ylabel("GAD-7 Score")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We:
 - Examined dataset metadata and structure, referencing all entities via their `@id`
 - Extracted main record sets and analyzed key numeric fields
 - Applied basic EDA: filtering, normalizing, and grouping (using field `@id`s)
 - Visualized distributions and relationships

This establishes a reproducible, standards-based workflow for FAIR and AI-ready survey data analysis.